# MedBot - AI Medical Assistant

## Deep Learning with Python - Final Project

This notebook demonstrates the technical implementation of MedBot, a RAG-based medical assistant.

### Contents
1. Environment Setup
2. Data Processing
3. Embedding Model & Vector Database
4. RAG Pipeline
5. LLM Integration
6. Demo

## 1. Environment Setup

In [ ]:
# Install dependencies
# !pip install -r ../requirements.txt

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from dotenv import load_dotenv

load_dotenv('../.env')
print("Environment loaded successfully!")

## 2. Data Processing

We use three datasets:
- **MedQuAD**: Medical question-answering pairs from NIH
- **FDA Drug Labels**: Medication information from OpenFDA
- **MTSamples**: Medical transcription samples

In [ ]:
# Sample data structure
sample_data = [
    {
        "question": "What are the symptoms of diabetes?",
        "answer": "Common symptoms include increased thirst, frequent urination, hunger, fatigue, and blurred vision.",
        "source": "MedQuAD"
    }
]
print("Sample data structure:")
print(sample_data)

## 3. Embedding Model & Vector Database

### Deep Learning Component
We use **Sentence Transformers** (based on BERT architecture) to convert text into dense vector representations.

In [ ]:
# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded: all-MiniLM-L6-v2")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
# Demonstrate embedding
sample_text = "I have a headache and feel dizzy"
embedding = model.encode(sample_text)

print(f"Input text: '{sample_text}'")
print(f"Embedding shape: {embedding.shape}")
print(f"First 10 values: {embedding[:10]}")

In [ ]:
# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="../vectorstore")
print("ChromaDB initialized")
print(f"Existing collections: {chroma_client.list_collections()}")

## 4. RAG Pipeline

Retrieval-Augmented Generation combines:
1. **Retrieval**: Find relevant documents using semantic similarity
2. **Generation**: Use LLM to generate response based on retrieved context

In [ ]:
from src.retriever import retrieve, format_context, add_documents

# Demo: Add sample documents to collection
demo_docs = [
    "Headache can be caused by tension, migraine, or dehydration. Common treatments include rest, hydration, and pain relievers.",
    "Dizziness may indicate inner ear problems, low blood pressure, or dehydration. Seek medical attention if persistent.",
    "Fever is often a sign of infection. Rest and fluids are recommended. See a doctor if fever exceeds 103 F."
]

demo_metadata = [
    {"source": "MedQuAD-1"},
    {"source": "MedQuAD-2"},
    {"source": "MedQuAD-3"}
]

# Add to demo collection
add_documents("demo_symptoms", demo_docs, demo_metadata)
print("Demo documents added to vector store")

In [ ]:
# Demo: Retrieve relevant documents
query = "I have a headache and feel dizzy"
results = retrieve(query, "demo_symptoms", top_k=2)

print(f"Query: '{query}'\n")
print("Retrieved documents:")
for i, (doc, meta, dist) in enumerate(zip(results['documents'], results['metadatas'], results['distances'])):
    print(f"\n[{i+1}] Distance: {dist:.4f}")
    print(f"Source: {meta['source']}")
    print(f"Content: {doc}")

## 5. LLM Integration

We use DeepSeek V3 API to generate responses based on retrieved context.

In [ ]:
from src.llm import get_response, build_messages
from src.prompts import SYMPTOM_PROMPT

# Build context from retrieved documents
context = format_context(results)
print("Formatted context:")
print(context)

In [ ]:
# Generate response with RAG
messages = build_messages(SYMPTOM_PROMPT, query, context)

print("Sending to LLM...")
response = get_response(messages)

print("\n" + "="*50)
print("MedBot Response:")
print("="*50)
print(response)

## 6. Full Demo

Complete RAG pipeline demonstration.

In [ ]:
def medbot_query(question: str, collection: str = "demo_symptoms"):
    """Complete RAG pipeline for MedBot."""
    print(f"Question: {question}\n")
    
    # Step 1: Retrieve
    print("Step 1: Retrieving relevant documents...")
    results = retrieve(question, collection, top_k=3)
    context = format_context(results)
    print(f"Found {len(results['documents'])} relevant documents\n")
    
    # Step 2: Build prompt
    print("Step 2: Building prompt with context...")
    messages = build_messages(SYMPTOM_PROMPT, question, context)
    
    # Step 3: Generate
    print("Step 3: Generating response...\n")
    response = get_response(messages)
    
    print("="*50)
    print("MedBot Response:")
    print("="*50)
    print(response)
    return response

In [ ]:
# Test the complete pipeline
medbot_query("What should I do if I have a fever?")

## Summary

### Deep Learning Components Used:
1. **Sentence Transformers (BERT-based)**: Text embedding for semantic similarity
2. **Vector Search**: Efficient nearest-neighbor retrieval
3. **Large Language Model**: Context-aware response generation

### Architecture:
```
User Query -> Embedding -> Vector Search -> Context Retrieval -> LLM -> Response
```

### Key Technologies:
- Sentence Transformers (all-MiniLM-L6-v2)
- ChromaDB (Vector Database)
- DeepSeek V3 (LLM)
- Gradio (Web Interface)